# 一貫性分析：ラップ間のばらつき

このノートブックでは、全ラップにわたるブレーキングポイント、コーナー速度、スロットル操作のばらつきを測定して、ドライビングの一貫性を分析します。

## このノートブックの内容

- **ブレーキングポイントの一貫性**: 各コーナーでのブレーキング開始位置のばらつきを示す箱ひげ図
- **コーナー速度の一貫性**: 各コーナーを通過する最低速度のばらつきを示す箱ひげ図
- **脱出速度の一貫性**: コーナー脱出速度のばらつきを示す箱ひげ図 — 脱出速度のばらつきは続くストレートでのタイムロスに直結
- **脱出速度オポチュニティ分析**: 脱出速度のばらつきがラップタイムに与える影響を、後続の加速ゾーン長を考慮してコーナーごとにランク付け
- **アクセル踏み込み点**: コーナー脱出時にフルスロットルに達する横Gを、コーナーのピーク横Gに対するパーセンテージで表示（英語: Throttle Acceptance / スロットルアクセプタンス）
- **サマリー統計テーブル**: 各セグメントの平均、標準偏差、最小値、最大値、範囲

## 結果の解釈方法

- **狭い箱ひげ図**（小さな範囲）: 一貫したパフォーマンス - 毎ラップ同じマークを狙えている
- **広い箱ひげ図**（大きな範囲）: 一貫性がない - 練習による改善の余地あり
- **外れ値**（ひげの外の点）: 異常なラップ - ミス、トラフィック、またはラインの試行錯誤
- **高い標準偏差**: 練習すべき重点項目
- **高いアクセル踏み込み点%**: 高い横Gを維持しながらより早くフルスロットルに到達 - より攻撃的な脱出

## 自分のデータを使用する場合

自分のデータを分析するには：

1. 下の**最初のセルを実行**してパッケージをインストールし、アップロードウィジェットを表示
2. **「Choose File」をクリック**して`.xrk`、`.xrz`、または`.ibt`ファイルを選択
3. **残りのセルをすべて実行**してデータを分析

ステータスインジケーターに使用中のファイルが表示されます。ファイルをアップロードしない場合は、サンプルデータが使用されます。

ノートブックは自動的に：
- GPSとペダルデータからコーナーとゾーンを検出
- すべての有効なラップを分析（ピットラップを除く）
- 各コースセグメントの一貫性指標を生成

## 必要なチャンネル

- GPSデータチャンネル（`GPS Latitude`、`GPS Longitude`、`GPS Speed`）
- ブレーキ圧（`BrakePress`）とスロットル（`PPS`）
- アクセル踏み込み点分析用の横加速度（`LateralAcc`）
- 有意義な一貫性分析のための複数ラップのデータ

**注意:** このノートブックはJupyterLite（ブラウザ）と通常のJupyterLab環境の両方で動作します。

In [1]:
# 必要なパッケージをインストール（JupyterLiteで必要、通常のJupyterLabでは既にインストール済みならスキップ）
%pip install -q motorsports-data-notebook

# Rustパーサーバックエンドを使用（ファイル読み込みが約3倍高速）
import os

os.environ["LIBXRK_BACKEND"] = "rust"

# コアライブラリをインポート
import pandas as pd
from IPython.display import display

# 可視化ライブラリ
import plotly.express as px
import plotly.graph_objects as go

# ヘルパー関数をインポート
from motorsports_data_notebook.channels import (
    get_best_lap_channels,
    get_top_laps,
)
from motorsports_data_notebook.corners import identify_corners
from motorsports_data_notebook.driver_analysis import find_throttle_acceptance
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_track_segments,
    visualize_throttle_acceptance,
    show_fig,
)
from motorsports_data_notebook.widgets import SessionPicker
from motorsports_data_notebook.zones import (
    compute_segment_stats,
    create_track_segments,
    detect_zones_averaged,
    get_corner_data,
)

# セッションピッカーとチャンネル設定
# 自分の.xrk/.xrz/.ibtファイルをアップロードするか、サンプルデータを使用
session = SessionPicker(
    default_file="../data/CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz",
    channel_mapping={
        # GPSチャンネル（コーナー検出に必要）
        "gps_latitude": "GPS Latitude",
        "gps_longitude": "GPS Longitude",
        "gps_speed": "GPS Speed",  # GPSからの速度（m/s）
        # ペダル入力（ゾーン検出に必要）
        "throttle": "PPS",  # スロットルポジションセンサー（0-100%）
        "brake": "BrakePress",  # ブレーキ圧（0-100%）
        # 車両ダイナミクス（アクセル踏み込み点分析に必要）
        "lateral_g": "LateralAcc",  # 横G
        "steering": "SteerAngle",  # ステアリング角度（度）
    },
)
session.display()

/home/runner/work/motorsports_data_notebook/motorsports_data_notebook/.venv/bin/python3: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# 読み込んだセッションデータを取得
log = session.get_log()
laps = session.get_laps()
CHANNEL_NAMES = session.get_channel_names()

In [3]:
# ラップタイム一覧を表示
laps.style.format({"lap_time": format_lap_time})  # type: ignore[dict-item]

,num,start_time,end_time,lap_type,lap_time
0,1,150454,279602,full,2:09.148
1,2,279602,406240,full,2:06.638
2,3,406240,532797,full,2:06.557
3,4,532797,659283,full,2:06.486
4,5,659283,787773,full,2:08.490
5,6,787773,913776,full,2:06.003
6,7,913776,1041398,full,2:07.622
7,8,1041398,1168323,full,2:06.925
8,9,1168323,1294676,full,2:06.353
9,10,1294676,1420573,full,2:05.897


In [4]:
# libxrk 0.5.0のメソッドを使用してベストラップのチャンネルデータを抽出
best_lap, channels = get_best_lap_channels(
    log, laps, [CHANNEL_NAMES["gps_latitude"], CHANNEL_NAMES["gps_longitude"], "distance_m"]
)

# ベストラップでフィルタし、コーナー検出用にGPS時間軸にリサンプル
best_lap_num = int(best_lap["num"])
gps_lat_ch = CHANNEL_NAMES["gps_latitude"]
gps_lon_ch = CHANNEL_NAMES["gps_longitude"]
aligned = (
    log.filter_by_lap(best_lap_num)
    .select_channels([gps_lat_ch, gps_lon_ch, "distance_m"])
    .resample_to_channel(gps_lat_ch)
    .channels
)

# コーナー検出用に配列に変換
lap_channels = {
    "GPS Latitude": aligned[gps_lat_ch].column(gps_lat_ch).to_numpy(),
    "GPS Longitude": aligned[gps_lon_ch].column(gps_lon_ch).to_numpy(),
    "distance_m": aligned["distance_m"].column("distance_m").to_numpy(),
}

In [5]:
# GPS座標から直接コーナーを識別
corners = identify_corners(
    lat=lap_channels["GPS Latitude"],
    lon=lap_channels["GPS Longitude"],
    threshold=0.003,  # 富士と袖ヶ浦のデータで調整済
    min_corner_length=15,
    min_gap=80,
)

print(f"{len(corners)}個のコーナーを検出")

10個のコーナーを検出


In [6]:
# 上位ラップを取得し、それら全体で平均化したゾーンを検出
top_laps = get_top_laps(laps, threshold_pct=1.03)

# 上位ラップ全体でブレーキング/加速ゾーンを検出・平均化
braking_zones, accel_zones = detect_zones_averaged(log, top_laps, CHANNEL_NAMES)

print(f"{len(braking_zones)}個のブレーキングゾーンと{len(accel_zones)}個の加速ゾーンを検出")

7個のブレーキングゾーンと9個の加速ゾーンを検出


In [7]:
# コースセグメントを作成
track_length = lap_channels["distance_m"][-1]
segments = create_track_segments(corners, braking_zones, accel_zones, track_length)

print(f"{len(segments)}個のコースセグメントを作成")

30個のコースセグメントを作成


In [8]:
# コーナー名付きのコースマップ（参照用）
lap_channels_df = pd.DataFrame(lap_channels)
fig = plot_track_segments(lap_channels_df, segments, title="コーナー名付きコースマップ")
show_fig(fig)

In [9]:
# 各ラップのセグメント統計を計算
stats_df = compute_segment_stats(log, top_laps, segments, CHANNEL_NAMES)

print(f"{len(top_laps)}ラップを分析中（ベストタイムの103%以内）...")
print(f"全ラップで{len(stats_df)}個のセグメント統計を計算")

13ラップを分析中（ベストタイムの103%以内）...
全ラップで390個のセグメント統計を計算


In [10]:
# ブレーキングの一貫性を可視化
# 各コーナーのブレーキングポイントのばらつきを平均を中心に表示

braking_stats = stats_df[stats_df["segment_type"] == "braking"].dropna(subset=["braking_point"])

if len(braking_stats) > 0:
    # 各セグメントの平均ブレーキングポイントからの偏差を計算
    braking_stats = braking_stats.copy()
    braking_stats["braking_deviation"] = braking_stats.groupby("segment_name")[
        "braking_point"
    ].transform(lambda x: x - x.mean())

    fig = px.box(
        braking_stats,
        x="segment_name",
        y="braking_deviation",
        title="コーナー別ブレーキングポイントの一貫性（平均を中心に）",
        labels={"braking_deviation": "平均からの偏差 (m)", "segment_name": "コーナー"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    # ゼロ（平均）に基準線を追加
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    show_fig(fig)
else:
    print("ブレーキングデータがありません")

In [11]:
# コーナー最低速度の一貫性を可視化
corner_stats = stats_df[stats_df["segment_type"] == "corner"].dropna(subset=["min_speed"])

if len(corner_stats) > 0:
    fig = px.box(
        corner_stats,
        x="segment_name",
        y="min_speed",
        title="コーナー最低速度の一貫性",
        labels={"min_speed": "最低速度 (km/h)", "segment_name": "コーナー"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("コーナー速度データがありません")

In [12]:
# コーナー脱出速度の一貫性を可視化
exit_speed_stats = stats_df[stats_df["segment_type"] == "corner"].dropna(subset=["exit_speed"])

if len(exit_speed_stats) > 0:
    fig = px.box(
        exit_speed_stats,
        x="segment_name",
        y="exit_speed",
        title="コーナー脱出速度の一貫性",
        labels={"exit_speed": "脱出速度 (km/h)", "segment_name": "コーナー"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("脱出速度データがありません")

In [13]:
# 脱出速度オポチュニティ分析
# 脱出速度の標準偏差 × 加速ゾーン長でコーナーをランク付け — 高いほどタイムロスが大きい

if len(exit_speed_stats) > 0:
    # コーナーごとの脱出速度統計を計算
    exit_grouped = (
        exit_speed_stats.groupby(["segment_name", "corner_id"])["exit_speed"]
        .agg(["std", "mean"])
        .reset_index()
    )
    exit_grouped.columns = ["segment_name", "corner_id", "exit_speed_std", "exit_speed_mean"]

    # 各コーナーを加速ゾーン長にマッピング
    accel_segments = {
        seg.corner_id: seg.end_dist - seg.start_dist
        for seg in segments
        if seg.segment_type == "acceleration" and seg.corner_id is not None
    }
    exit_grouped["accel_zone_length_m"] = exit_grouped["corner_id"].map(accel_segments)
    exit_grouped = exit_grouped.dropna(subset=["accel_zone_length_m", "exit_speed_std"])

    # オポチュニティスコアを計算
    exit_grouped["opportunity_score"] = (
        exit_grouped["exit_speed_std"] * exit_grouped["accel_zone_length_m"]
    )
    exit_grouped = exit_grouped.sort_values("opportunity_score", ascending=False)

    print("脱出速度オポチュニティ分析")
    print("（スコアが高いほどラップタイムのロスが大きい）\n")
    display(
        exit_grouped[
            [
                "segment_name",
                "exit_speed_mean",
                "exit_speed_std",
                "accel_zone_length_m",
                "opportunity_score",
            ]
        ]
        .rename(
            columns={
                "segment_name": "コーナー",
                "exit_speed_mean": "平均脱出速度 (km/h)",
                "exit_speed_std": "脱出速度 標準偏差 (km/h)",
                "accel_zone_length_m": "加速ゾーン長 (m)",
                "opportunity_score": "オポチュニティスコア",
            }
        )
        .reset_index(drop=True)
        .style.format(
            {
                "平均脱出速度 (km/h)": "{:.1f}",
                "脱出速度 標準偏差 (km/h)": "{:.2f}",
                "加速ゾーン長 (m)": "{:.0f}",
                "オポチュニティスコア": "{:.1f}",
            }
        )
    )
else:
    print("オポチュニティ分析に使用できる脱出速度データがありません")

脱出速度オポチュニティ分析
（スコアが高いほどラップタイムのロスが大きい）



,コーナー,平均脱出速度 (km/h),脱出速度 標準偏差 (km/h),加速ゾーン長 (m),オポチュニティスコア
0,Turn 10,99.0,2.14,761,1627.0
1,Turn 4,117.0,2.52,528,1330.4
2,Turn 6,64.8,3.72,206,765.3
3,Turn 2,135.7,3.95,166,658.0
4,Turn 5,150.4,1.79,306,546.5
5,Turn 7,87.7,4.12,122,501.0
6,Turn 9,95.7,3.32,97,321.1
7,Turn 1,131.9,1.26,232,292.1
8,Turn 8,105.1,2.16,80,172.6
9,Turn 3,139.9,2.70,19,51.0


In [14]:
# 脱出速度オポチュニティ棒グラフ
if len(exit_speed_stats) > 0 and len(exit_grouped) > 0:
    plot_data = exit_grouped.sort_values("opportunity_score", ascending=True)

    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            y=plot_data["segment_name"],
            x=plot_data["opportunity_score"],
            orientation="h",
            text=[
                f"\u03c3={std:.1f} km/h, {length:.0f}m"
                for std, length in zip(
                    plot_data["exit_speed_std"], plot_data["accel_zone_length_m"]
                )
            ],
            textposition="auto",
            hovertext=[
                f"{name}<br>脱出速度 標準偏差: {std:.2f} km/h<br>"
                f"加速ゾーン: {length:.0f}m<br>スコア: {score:.1f}"
                for name, std, length, score in zip(
                    plot_data["segment_name"],
                    plot_data["exit_speed_std"],
                    plot_data["accel_zone_length_m"],
                    plot_data["opportunity_score"],
                )
            ],
            hoverinfo="text",
        )
    )
    fig.update_layout(
        title="脱出速度オポチュニティ（高いほどタイムロスが大きい）",
        xaxis_title="オポチュニティスコア（脱出速度 標準偏差 × 加速ゾーン長）",
        yaxis_title="コーナー",
        width=900,
        height=max(400, len(plot_data) * 50 + 100),
    )
    show_fig(fig)

In [15]:
# 全ラップにわたる各コーナーのアクセル踏み込み点を計算
# アクセル踏み込み点 = 持続的なフルスロットル時の横G / コーナーのピーク横G

# 横G計算と可視化のスムージングウィンドウ
LATERAL_G_SMOOTHING_WINDOW = 1

# アクセル踏み込み点分析に必要なチャンネル
throttle_ch = CHANNEL_NAMES["throttle"]
lateral_g_ch = CHANNEL_NAMES["lateral_g"]
THROTTLE_ACCEPTANCE_CHANNELS = ["distance_m", throttle_ch, lateral_g_ch]

throttle_acceptance_stats = []

for idx, lap in top_laps.iterrows():
    lap_num = int(lap["num"])

    # libxrk 0.5.0のメソッドを使用してラップでフィルタ、チャンネル選択、リサンプル
    aligned = (
        log.filter_by_lap(lap_num)
        .select_channels(THROTTLE_ACCEPTANCE_CHANNELS)
        .resample_to_channel("distance_m")
        .channels
    )

    # チャンネル + タイムコードでDataFrameを構築（タイムコードはアラインされたテーブルから取得）
    lap_data = pd.DataFrame(
        {name: aligned[name].column(name).to_numpy() for name in THROTTLE_ACCEPTANCE_CHANNELS}
    )
    # 参照チャンネルのテーブルからタイムコードを追加
    lap_data["timecodes"] = aligned["distance_m"].column("timecodes").to_numpy()

    if len(lap_data) < 10:
        continue

    for corner in corners:
        result = find_throttle_acceptance(
            lap_data,
            corner,
            CHANNEL_NAMES,
            smoothing_window=LATERAL_G_SMOOTHING_WINDOW,
        )
        if result is not None:
            throttle_acceptance_stats.append(
                {
                    "corner_name": corner.name,
                    "corner_id": corner.id,
                    "lap_num": lap["num"],
                    "throttle_acceptance_pct": result["throttle_acceptance_pct"],
                    "lateral_g_at_throttle": result["lateral_g_at_throttle"],
                    "peak_lateral_g": result["peak_lateral_g"],
                    "full_throttle_dist": result["full_throttle_dist"],
                }
            )

throttle_acceptance_df = pd.DataFrame(throttle_acceptance_stats)
print(f"{len(throttle_acceptance_df)}個のコーナー/ラップ組み合わせでアクセル踏み込み点を計算")

127個のコーナー/ラップ組み合わせでアクセル踏み込み点を計算


In [16]:
# アクセル踏み込み点の一貫性を可視化
# ドライバーがピーク横Gの何パーセントでフルスロットルに達するかを表示

if len(throttle_acceptance_df) > 0:
    corner_order = [c.name for c in corners]
    fig = px.box(
        throttle_acceptance_df,
        x="corner_name",
        y="throttle_acceptance_pct",
        title="コーナー別アクセル踏み込み点（フルスロットル時のピーク横G%）",
        labels={
            "throttle_acceptance_pct": "アクセル踏み込み点 (%)",
            "corner_name": "コーナー",
        },
        category_orders={"corner_name": corner_order},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    # 基準線を追加
    fig.add_hline(
        y=100,
        line_dash="dash",
        line_color="red",
        opacity=0.5,
        annotation_text="100% = ピークGでフルスロットル",
    )
    show_fig(fig)
else:
    print("アクセル踏み込み点データがありません")

In [17]:
# アクセル踏み込み点のサマリー統計テーブル
if len(throttle_acceptance_df) > 0:
    throttle_summary = []
    for corner_name in throttle_acceptance_df["corner_name"].unique():
        corner_data = throttle_acceptance_df[throttle_acceptance_df["corner_name"] == corner_name]
        throttle_summary.append(
            {
                "コーナー": corner_name,
                "平均 (%)": corner_data["throttle_acceptance_pct"].mean(),
                "標準偏差 (%)": corner_data["throttle_acceptance_pct"].std(),
                "最小 (%)": corner_data["throttle_acceptance_pct"].min(),
                "最大 (%)": corner_data["throttle_acceptance_pct"].max(),
                "範囲 (%)": corner_data["throttle_acceptance_pct"].max()
                - corner_data["throttle_acceptance_pct"].min(),
                "N": len(corner_data),
            }
        )

    throttle_summary_df = pd.DataFrame(throttle_summary)
    display(
        throttle_summary_df.style.format(
            {
                "平均 (%)": "{:.1f}",
                "標準偏差 (%)": "{:.1f}",
                "最小 (%)": "{:.1f}",
                "最大 (%)": "{:.1f}",
                "範囲 (%)": "{:.1f}",
            }
        )
    )
else:
    print("アクセル踏み込み点データがありません")

,コーナー,平均 (%),標準偏差 (%),最小 (%),最大 (%),範囲 (%),N
0,Turn 1,77.7,17.9,34.8,97.9,63.2,13
1,Turn 2,72.6,11.5,51.6,95.9,44.3,13
2,Turn 3,79.8,6.2,69.5,89.5,19.9,13
3,Turn 4,77.3,9.2,61.9,91.6,29.6,13
4,Turn 5,86.1,5.7,76.1,95.3,19.1,13
5,Turn 7,79.9,11.2,53.2,92.8,39.5,13
6,Turn 8,69.3,6.4,57.8,81.9,24.1,13
7,Turn 9,51.0,26.8,3.8,86.5,82.7,12
8,Turn 10,65.9,9.5,44.7,76.8,32.1,13
9,Turn 6,83.6,9.6,64.5,92.5,28.0,11


In [18]:
# ターン1（ベストラップ）のアクセル踏み込み点の概念を可視化
# スロットル、ブレーキ、ステアリング、横Gを距離に対して基準線付きで表示

turn1 = corners[0]

# libxrk 0.5.0のメソッドを使用してベストラップのチャンネルを取得
best_lap_num = int(best_lap["num"])
best_lap_aligned = (
    log.filter_by_lap(best_lap_num)
    .select_channels(THROTTLE_ACCEPTANCE_CHANNELS)
    .resample_to_channel("distance_m")
    .channels
)
best_lap_df = pd.DataFrame(
    {name: best_lap_aligned[name].column(name).to_numpy() for name in THROTTLE_ACCEPTANCE_CHANNELS}
)
# 参照チャンネルのテーブルからタイムコードを追加
best_lap_df["timecodes"] = best_lap_aligned["distance_m"].column("timecodes").to_numpy()

turn1_result = find_throttle_acceptance(
    best_lap_df, turn1, CHANNEL_NAMES, smoothing_window=LATERAL_G_SMOOTHING_WINDOW
)
assert turn1_result is not None, f"{turn1.name}のアクセル踏み込み点を計算できませんでした"

# ベストラップのコーナーデータを取得（フィルタ済みのログを受け取る）
brake_ch = CHANNEL_NAMES["brake"]
steering_ch = CHANNEL_NAMES["steering"]
CORNER_VIZ_CHANNELS = ["distance_m", throttle_ch, brake_ch, lateral_g_ch, steering_ch]
lap_log = log.filter_by_lap(best_lap_num)
corner_data = get_corner_data(lap_log, turn1, CORNER_VIZ_CHANNELS, margin=50)

# スムージングした横Gを計算
lateral_g_smooth = (
    corner_data[lateral_g_ch]
    .abs()
    .rolling(window=LATERAL_G_SMOOTHING_WINDOW, center=True, min_periods=1)
    .mean()
)

fig = visualize_throttle_acceptance(
    distance=corner_data["distance_m"],
    throttle=corner_data[throttle_ch],
    lateral_g=lateral_g_smooth,
    corner=turn1,
    throttle_acceptance_result=turn1_result,
    brake=corner_data.get(brake_ch),
    steering=corner_data.get(steering_ch),
)
show_fig(fig)

In [19]:
# サマリー統計テーブル
def compute_summary_stats(stats_df):
    """全ラップにわたる各セグメントのサマリー統計を計算"""
    summary = []

    # ブレーキングセグメント
    for seg_name in stats_df[stats_df["segment_type"] == "braking"]["segment_name"].unique():
        seg_data = stats_df[
            (stats_df["segment_name"] == seg_name) & stats_df["braking_point"].notna()
        ]
        if len(seg_data) > 0:
            summary.append(
                {
                    "セグメント": seg_name,
                    "タイプ": "ブレーキング",
                    "指標": "ブレーキングポイント (m)",
                    "平均": seg_data["braking_point"].mean(),
                    "標準偏差": seg_data["braking_point"].std(),
                    "最小": seg_data["braking_point"].min(),
                    "最大": seg_data["braking_point"].max(),
                    "範囲": seg_data["braking_point"].max() - seg_data["braking_point"].min(),
                    "N": len(seg_data),
                }
            )

    # コーナーセグメント
    for seg_name in stats_df[stats_df["segment_type"] == "corner"]["segment_name"].unique():
        seg_data = stats_df[(stats_df["segment_name"] == seg_name) & stats_df["min_speed"].notna()]
        if len(seg_data) > 0:
            summary.append(
                {
                    "セグメント": seg_name,
                    "タイプ": "コーナー",
                    "指標": "最低速度 (km/h)",
                    "平均": seg_data["min_speed"].mean(),
                    "標準偏差": seg_data["min_speed"].std(),
                    "最小": seg_data["min_speed"].min(),
                    "最大": seg_data["min_speed"].max(),
                    "範囲": seg_data["min_speed"].max() - seg_data["min_speed"].min(),
                    "N": len(seg_data),
                }
            )

        exit_data = stats_df[
            (stats_df["segment_name"] == seg_name) & stats_df["exit_speed"].notna()
        ]
        if len(exit_data) > 0:
            summary.append(
                {
                    "セグメント": seg_name,
                    "タイプ": "コーナー",
                    "指標": "脱出速度 (km/h)",
                    "平均": exit_data["exit_speed"].mean(),
                    "標準偏差": exit_data["exit_speed"].std(),
                    "最小": exit_data["exit_speed"].min(),
                    "最大": exit_data["exit_speed"].max(),
                    "範囲": exit_data["exit_speed"].max() - exit_data["exit_speed"].min(),
                    "N": len(exit_data),
                }
            )

    return pd.DataFrame(summary)


summary_df = compute_summary_stats(stats_df)
summary_df.style.format(
    {"平均": "{:.1f}", "標準偏差": "{:.1f}", "最小": "{:.1f}", "最大": "{:.1f}", "範囲": "{:.1f}"}
)

,セグメント,タイプ,指標,平均,標準偏差,最小,最大,範囲,N
0,Turn 1 Braking,ブレーキング,ブレーキングポイント (m),576.7,5.0,572.4,585.7,13.3,13
1,Turn 2 Braking,ブレーキング,ブレーキングポイント (m),1214.8,4.6,1211.0,1224.8,13.7,13
2,Turn 4 Braking,ブレーキング,ブレーキングポイント (m),1903.9,3.3,1901.4,1912.8,11.4,13
3,Turn 6 Braking,ブレーキング,ブレーキングポイント (m),2686.3,2.3,2684.1,2692.1,8.0,13
4,Turn 7 Braking,ブレーキング,ブレーキングポイント (m),2686.3,2.3,2684.1,2692.1,8.0,13
5,Turn 1,コーナー,最低速度 (km/h),64.3,2.7,58.3,68.4,10.1,13
6,Turn 1,コーナー,脱出速度 (km/h),131.9,1.3,129.8,133.9,4.1,13
7,Turn 2,コーナー,最低速度 (km/h),127.7,4.6,115.8,132.7,16.9,13
8,Turn 2,コーナー,脱出速度 (km/h),135.7,4.0,127.3,141.1,13.8,13
9,Turn 3,コーナー,最低速度 (km/h),136.7,2.8,131.1,142.1,11.0,13
